[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/paper_implementation/blob/main/mean_flow_sanity_check.ipynb)

# MeanFlow sanity check — MNIST + ~4M DiT, 20k steps

이 노트북은 **MeanFlow 자체가 작은 이미지 문제에서 학습되는지 먼저 검증**하기 위한 독립 sanity check다.

공식 MeanFlow 저장소가 연결하는 `pkulwj1994/easy_meanflow`의 MNIST Colab recipe를 기준으로 한다. 원 Colab의 U-Net은 약 3.9M parameter이며 20,000 step을 학습한다. 여기서는 **데이터와 MeanFlow loss/optimizer recipe를 유지하고, backbone만 비슷한 체급의 DiT로 바꾼다.**

- MNIST 28×28 → pad 2 → **32×32**
- DiT: patch 4, hidden 224, depth 4, heads 8 → 약 **3.8M parameters**
- batch 128, **20,000 steps**
- Adam, lr=1e-3, betas=(0.9, 0.99), eps=1e-8, weight decay 없음
- MeanFlow: logit-normal(-0.4, 1), batch 75%에서 r=t, condition=(t, h=t-r)
- adaptive loss: linked MNIST Colab과 같은 norm_eps=1.0

TensorBoard 대신 결과를 바로 전달하기 쉽게 현재 run 폴더에 다음 파일을 계속 갱신한다.

1. `samples_progress.png` — 같은 고정 noise에서 step 0 / 2k / 5k / 10k / 15k / 20k 생성 결과
2. `training_summary.png` — 핵심 학습/내부 진단 곡선 한 장
3. `metrics.csv` — 250 step 간격의 모든 숫자

추가로 20k 학습 비용을 버리지 않도록 마지막 model checkpoint도 저장한다.


## 0. Setup

Colab **T4 GPU** 기준이다. 한 run의 결과는 별도 폴더에 저장된다.


In [ ]:
# @title 0-1. Imports / configuration / run directory
import copy
import csv
import math
import os
import random
import time
import warnings
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.func import jvp
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("Select a Colab T4 GPU runtime first.")

GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
if "T4" not in GPU_NAME:
    warnings.warn(f"Sized for T4, but current GPU is {GPU_NAME}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.mha.set_fastpath_enabled(False)

TRAIN_STEPS = 20_000
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
ADAM_BETAS = (0.9, 0.99)
ADAM_EPS = 1e-8
DIAGNOSTIC_EVERY = 250
DIAGNOSTIC_BATCH_SIZE = 64
FIXED_SAMPLE_COUNT = 16
SAMPLE_STEPS = {0, 2_000, 5_000, 10_000, 15_000, 20_000}

P_MEAN = -0.4
P_STD = 1.0
DATA_PROPORTION = 0.75
NORM_P = 1.0
NORM_EPS = 1.0

ROOT_DIR = "/content/meanflow_mnist_dit_sanity"
RUN_NAME = "mnist_dit20k_" + time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(ROOT_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

SAMPLES_PATH = os.path.join(RUN_DIR, "samples_progress.png")
SUMMARY_PATH = os.path.join(RUN_DIR, "training_summary.png")
METRICS_PATH = os.path.join(RUN_DIR, "metrics.csv")
CHECKPOINT_PATH = os.path.join(RUN_DIR, "meanflow_dit_mnist_20k.pt")

print("RUN_DIR:", RUN_DIR)


## 1. MNIST — linked Colab과 같은 32×32 입력

연결된 MNIST Colab은 `ToTensor → Pad(2) → Normalize(0.5, 0.5)`를 사용한다. 따라서 28×28 숫자를 32×32로 만들고 [-1,1] 범위로 정규화한다.


In [ ]:
# @title 1-1. Download MNIST and create loaders
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Pad(2),
        transforms.Normalize((0.5,), (0.5,)),
    ]
)

train_dataset = datasets.MNIST(
    root="/content/mnist_data",
    train=True,
    transform=transform,
    download=True,
)
test_dataset = datasets.MNIST(
    root="/content/mnist_data",
    train=False,
    transform=transform,
    download=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=DIAGNOSTIC_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

images, labels = next(iter(train_loader))
print("train size:", len(train_dataset))
print("test size:", len(test_dataset))
print("batch:", images.shape, "range:", images.min().item(), images.max().item())


## 2. DiT backbone — U-Net과 비슷한 parameter budget

linked MNIST Colab의 U-Net은 약 3.9M parameters다. 여기서는 patch 4, hidden 224, depth 4, heads 8인 DiT를 사용해 약 3.8M으로 맞춘다.

MeanFlow condition은 두 개의 **독립된** scalar embedding으로 `t`와 `h=t-r`를 넣는다. DiT block은 adaLN-Zero 방식으로 시작해 초기 residual update가 0에 가깝도록 한다.

PyTorch attention fastpath는 forward-mode AD(JVP)와 충돌할 수 있어 math attention 경로를 사용한다.


In [ ]:
# @title 2-1. ~3.8M MeanFlow DiT
class ScalarEmbed(nn.Module):
    def __init__(self, dim, fourier=64):
        super().__init__()
        self.fourier = fourier
        self.mlp = nn.Sequential(
            nn.Linear(fourier, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )

    def forward(self, value):
        value = value.reshape(-1, 1)
        half = self.fourier // 2
        frequencies = torch.exp(
            torch.linspace(
                math.log(1.0),
                math.log(1000.0),
                half,
                device=value.device,
            )
        )
        angles = value * frequencies[None] * 2.0 * math.pi
        embedding = torch.cat([angles.sin(), angles.cos()], dim=1)
        return self.mlp(embedding)


def math_attention_context():
    if DEVICE == "cuda":
        return torch.backends.cuda.sdp_kernel(
            enable_flash=False,
            enable_math=True,
            enable_mem_efficient=False,
        )
    return nullcontext()


class DiTBlock(nn.Module):
    def __init__(self, dim=224, heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False)
        self.feed_forward = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )
        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(dim, 6 * dim),
        )
        nn.init.zeros_(self.modulation[-1].weight)
        nn.init.zeros_(self.modulation[-1].bias)

    def forward(self, tokens, condition):
        shift1, scale1, gate1, shift2, scale2, gate2 = (
            self.modulation(condition).chunk(6, dim=-1)
        )
        hidden = self.norm1(tokens) * (1.0 + scale1[:, None]) + shift1[:, None]
        with math_attention_context():
            attended, _ = self.attention(
                hidden, hidden, hidden, need_weights=True
            )
        tokens = tokens + gate1[:, None] * attended
        hidden = self.norm2(tokens) * (1.0 + scale2[:, None]) + shift2[:, None]
        return tokens + gate2[:, None] * self.feed_forward(hidden)


class MeanFlowDiT(nn.Module):
    def __init__(self, dim=224, depth=4, heads=8, patch=4):
        super().__init__()
        self.patch = patch
        self.grid = 32 // patch
        self.input_projection = nn.Conv2d(1, dim, patch, patch)
        self.position = nn.Parameter(
            torch.randn(1, self.grid * self.grid, dim) * 0.02
        )
        self.time_embed = ScalarEmbed(dim)
        self.interval_embed = ScalarEmbed(dim)
        self.blocks = nn.ModuleList(
            [DiTBlock(dim, heads) for _ in range(depth)]
        )
        self.final_norm = nn.LayerNorm(dim)
        self.final_linear = nn.Linear(dim, patch * patch)
        nn.init.zeros_(self.final_linear.weight)
        nn.init.zeros_(self.final_linear.bias)

    def _tokens_and_condition(self, images, t, h):
        tokens = (
            self.input_projection(images).flatten(2).transpose(1, 2)
            + self.position
        )
        condition = self.time_embed(t) + self.interval_embed(h)
        return tokens, condition

    def forward_features(self, images, t, h):
        tokens, condition = self._tokens_and_condition(images, t, h)
        features = []
        for block in self.blocks:
            tokens = block(tokens, condition)
            features.append(tokens)
        return features

    def forward(self, images, t, h):
        tokens, condition = self._tokens_and_condition(images, t, h)
        for block in self.blocks:
            tokens = block(tokens, condition)
        patches = self.final_linear(self.final_norm(tokens)).view(
            images.size(0), self.grid, self.grid, self.patch, self.patch
        )
        return patches.permute(0, 1, 3, 2, 4).reshape(
            images.size(0), 1, 32, 32
        )


model = MeanFlowDiT().to(DEVICE)
PARAMETERS = sum(p.numel() for p in model.parameters())
print("DiT parameters:", f"{PARAMETERS:,}")
print("target U-Net scale: about 3.9M")


## 3. MeanFlow loss — linked MNIST Colab recipe

기본 interpolation은

$$z_t=(1-t)x+t\epsilon,\qquad v=\epsilon-x.$$

두 시간은 logit-normal에서 뽑아 큰 값을 `t`, 작은 값을 `r`로 두고, batch의 75%에서 `r=t`로 바꾼다. network는 `u(z_t,t,h)`에서 `h=t-r`를 입력받는다.

MeanFlow target은

$$u_{\mathrm{tgt}}=v-(t-r)\frac{d}{dt}u_\theta,$$

이며 JVP tangent는 `(v, 1, 0)`이다. adaptive loss는 linked MNIST Colab과 똑같이 pixel SSE와 `norm_eps=1.0`을 쓴다.


In [ ]:
# @title 3-1. MeanFlow objective
def sample_logit_normal(shape, device):
    normal = torch.randn(shape, device=device)
    return torch.sigmoid(normal * P_STD + P_MEAN)


def sample_training_tuple(images):
    batch = images.size(0)
    shape = (batch, 1, 1, 1)
    t = sample_logit_normal(shape, images.device)
    r = sample_logit_normal(shape, images.device)
    t, r = torch.maximum(t, r), torch.minimum(t, r)

    zero_mask = torch.arange(batch, device=images.device) < int(
        batch * DATA_PROPORTION
    )
    zero_mask = zero_mask.view(shape)
    r = torch.where(zero_mask, t, r)

    noise = torch.randn_like(images)
    z_t = (1.0 - t) * images + t * noise
    velocity = noise - images
    return z_t, velocity, t, r


def meanflow_prediction_target(current_model, z_t, velocity, t, r):
    def u_wrapper(z_value, t_value, r_value):
        return current_model(
            z_value,
            t_value.squeeze(-1).squeeze(-1).squeeze(-1),
            (t_value - r_value).squeeze(-1).squeeze(-1).squeeze(-1),
        )

    prediction, du_dt = jvp(
        u_wrapper,
        (z_t, t, r),
        (velocity, torch.ones_like(t), torch.zeros_like(r)),
    )
    target = velocity - torch.clamp(t - r, 0.0, 1.0) * du_dt
    return prediction, target.detach()


def meanflow_loss(current_model, images):
    z_t, velocity, t, r = sample_training_tuple(images)
    prediction, target = meanflow_prediction_target(
        current_model, z_t, velocity, t, r
    )

    residual = prediction - target
    unweighted = residual.square().sum(dim=(1, 2, 3))
    with torch.no_grad():
        adaptive_weight = 1.0 / (unweighted + NORM_EPS).pow(NORM_P)
    loss = (unweighted * adaptive_weight).mean()

    return {
        "loss": loss,
        "raw_identity_mse": residual.square().mean().detach(),
        "velocity_mse": (prediction - velocity).square().mean().detach(),
    }


## 4. Fixed diagnostics and shareable outputs

학습 중 매번 같은 입력을 검사한다. 생성은 같은 noise 16개를 사용한다. 진단 batch도 고정한다.

`training_summary.png`에는 다음 여섯 패널을 넣는다.

- fixed raw identity MSE
- finite-interval target cosine (`r<t`만)
- `r=t` boundary velocity MSE
- gradient norm
- 평균 block weight drift
- 마지막 block hidden-feature effective rank

즉 생성 그림이 아직 불분명해도 parameter와 representation이 실제로 움직이는지 한 파일에서 확인할 수 있다.


In [ ]:
# @title 4-1. Fixed inputs and diagnostic helpers
fixed_generator = torch.Generator().manual_seed(SEED + 100)
fixed_noise = torch.randn(
    FIXED_SAMPLE_COUNT, 1, 32, 32, generator=fixed_generator
).to(DEVICE)

diagnostic_images, _ = next(iter(test_loader))
diagnostic_images = diagnostic_images[:DIAGNOSTIC_BATCH_SIZE].to(DEVICE)
diagnostic_noise = torch.randn(
    diagnostic_images.shape, generator=fixed_generator
).to(DEVICE)

normal1 = torch.randn(DIAGNOSTIC_BATCH_SIZE, generator=fixed_generator)
normal2 = torch.randn(DIAGNOSTIC_BATCH_SIZE, generator=fixed_generator)
time1 = torch.sigmoid(normal1 * P_STD + P_MEAN).to(DEVICE)
time2 = torch.sigmoid(normal2 * P_STD + P_MEAN).to(DEVICE)
diagnostic_t = torch.maximum(time1, time2).view(-1, 1, 1, 1)
diagnostic_r = torch.minimum(time1, time2).view(-1, 1, 1, 1)
count_equal = int(DIAGNOSTIC_BATCH_SIZE * DATA_PROPORTION)
diagnostic_r = diagnostic_r.clone()
diagnostic_r[:count_equal] = diagnostic_t[:count_equal]
interval_mask = (diagnostic_r < diagnostic_t).view(-1)

initial_blocks = [
    [parameter.detach().clone() for parameter in block.parameters()]
    for block in model.blocks
]

sample_snapshots = {}
metric_rows = []


@torch.no_grad()
def generate_one_step(current_model):
    current_model.eval()
    ones = torch.ones(FIXED_SAMPLE_COUNT, device=DEVICE)
    return fixed_noise - current_model(fixed_noise, ones, ones)


def total_grad_norm(current_model):
    squared = torch.zeros((), device=DEVICE)
    for parameter in current_model.parameters():
        if parameter.grad is not None:
            squared = squared + parameter.grad.detach().float().square().sum()
    return squared.sqrt().item()


def mean_block_drift(current_model):
    values = []
    for block, initial_parameters in zip(current_model.blocks, initial_blocks):
        numerator = torch.zeros((), device=DEVICE)
        denominator = torch.zeros((), device=DEVICE)
        for parameter, initial in zip(block.parameters(), initial_parameters):
            current = parameter.detach()
            numerator += (current - initial).square().sum()
            denominator += initial.to(DEVICE).square().sum()
        values.append(
            (numerator.sqrt() / denominator.sqrt().clamp_min(1e-12)).item()
        )
    return float(np.mean(values))


def effective_rank(matrix):
    matrix = matrix.float() - matrix.float().mean(dim=0, keepdim=True)
    singular_values = torch.linalg.svdvals(matrix)
    probabilities = singular_values / singular_values.sum().clamp_min(1e-12)
    entropy = -(probabilities * probabilities.clamp_min(1e-12).log()).sum()
    return entropy.exp().item()


@torch.no_grad()
def fixed_diagnostics(current_model):
    current_model.eval()
    z_t = (1.0 - diagnostic_t) * diagnostic_images + diagnostic_t * diagnostic_noise
    velocity = diagnostic_noise - diagnostic_images

    prediction, target = meanflow_prediction_target(
        current_model, z_t, velocity, diagnostic_t, diagnostic_r
    )
    raw_mse = (prediction - target).square().mean().item()
    cosine = F.cosine_similarity(
        prediction.flatten(1), target.flatten(1), dim=1, eps=1e-8
    )
    interval_cosine = cosine[interval_mask].mean().item()
    all_cosine = cosine.mean().item()

    t_flat = diagnostic_t.view(-1)
    boundary_prediction = current_model(
        z_t, t_flat, torch.zeros_like(t_flat)
    )
    boundary_mse = (boundary_prediction - velocity).square().mean().item()

    feature_t = torch.full((DIAGNOSTIC_BATCH_SIZE,), 0.5, device=DEVICE)
    feature_h = torch.full_like(feature_t, 0.5)
    feature_z = 0.5 * diagnostic_images + 0.5 * diagnostic_noise
    last_feature = current_model.forward_features(
        feature_z, feature_t, feature_h
    )[-1]
    feature_matrix = last_feature.reshape(-1, last_feature.size(-1))

    return {
        "fixed_raw_identity_mse": raw_mse,
        "fixed_target_cosine_all": all_cosine,
        "fixed_target_cosine_interval": interval_cosine,
        "boundary_velocity_mse": boundary_mse,
        "mean_block_weight_drift": mean_block_drift(current_model),
        "last_block_feature_std": last_feature.float().std().item(),
        "last_block_effective_rank": effective_rank(feature_matrix),
    }


def write_metrics_csv():
    if not metric_rows:
        return
    with open(METRICS_PATH, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=metric_rows[0].keys())
        writer.writeheader()
        writer.writerows(metric_rows)


def write_training_summary():
    if not metric_rows:
        return
    steps = [row["step"] for row in metric_rows]
    panels = [
        ("fixed_raw_identity_mse", "Fixed raw identity MSE", True),
        ("fixed_target_cosine_interval", "Finite-interval cosine", False),
        ("boundary_velocity_mse", "r=t boundary velocity MSE", True),
        ("grad_norm", "Gradient norm", True),
        ("mean_block_weight_drift", "Mean block weight drift", False),
        ("last_block_effective_rank", "Last-block effective rank", False),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for axis, (key, title, log_scale) in zip(axes.flat, panels):
        values = [row[key] for row in metric_rows]
        axis.plot(steps, values)
        axis.set_title(title)
        axis.set_xlabel("Step")
        axis.grid(True, alpha=0.25)
        if log_scale and all(np.isfinite(v) and v > 0 for v in values):
            axis.set_yscale("log")
    fig.tight_layout()
    fig.savefig(SUMMARY_PATH, dpi=150, bbox_inches="tight")
    plt.close(fig)


def write_samples_progress():
    if not sample_snapshots:
        return
    ordered = [step for step in sorted(SAMPLE_STEPS) if step in sample_snapshots]
    fig, axes = plt.subplots(3, 2, figsize=(10, 15))
    axes = axes.flat
    for axis in axes:
        axis.axis("off")
    for axis, step in zip(axes, ordered):
        images = sample_snapshots[step]
        grid = make_grid((images.clamp(-1, 1) + 1.0) / 2.0, nrow=4, padding=2)
        axis.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray", vmin=0, vmax=1)
        axis.set_title(f"step {step:,}")
        axis.axis("off")
    fig.tight_layout()
    fig.savefig(SAMPLES_PATH, dpi=180, bbox_inches="tight")
    plt.close(fig)


## 5. Train 20,000 steps

linked MNIST Colab처럼 **Adam + 20k step**을 사용한다. EMA, weight decay, scheduler는 넣지 않는다.

250 step마다 fixed diagnostics를 계산해서 `metrics.csv`와 `training_summary.png`를 갱신한다. 지정된 milestone에서는 `samples_progress.png`도 같은 파일명으로 갱신한다. 따라서 학습 도중에도 파일을 열어 진행 상태를 볼 수 있다.


In [ ]:
# @title 5-1. Train and continuously update the three shareable result files
def infinite_loader(loader):
    while True:
        for batch in loader:
            yield batch


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=ADAM_BETAS,
    eps=ADAM_EPS,
)
training_iterator = infinite_loader(train_loader)

# Step 0 baseline.
sample_snapshots[0] = generate_one_step(model).detach().cpu()
diagnostics = fixed_diagnostics(model)
metric_rows.append(
    {
        "step": 0,
        "train_adaptive_loss": float("nan"),
        "train_raw_identity_mse": float("nan"),
        "train_velocity_mse": float("nan"),
        "grad_norm": float("nan"),
        **diagnostics,
    }
)
write_metrics_csv()
write_training_summary()
write_samples_progress()

model.train()
start_time = time.time()

for step in range(1, TRAIN_STEPS + 1):
    images, _ = next(training_iterator)
    images = images.to(DEVICE, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)
    train_metrics = meanflow_loss(model, images)
    train_metrics["loss"].backward()
    grad_norm = total_grad_norm(model)
    optimizer.step()

    if step % DIAGNOSTIC_EVERY == 0 or step == TRAIN_STEPS:
        diagnostics = fixed_diagnostics(model)
        metric_rows.append(
            {
                "step": step,
                "train_adaptive_loss": train_metrics["loss"].item(),
                "train_raw_identity_mse": train_metrics["raw_identity_mse"].item(),
                "train_velocity_mse": train_metrics["velocity_mse"].item(),
                "grad_norm": grad_norm,
                **diagnostics,
            }
        )
        write_metrics_csv()
        write_training_summary()

        elapsed = (time.time() - start_time) / 60.0
        print(
            f"step={step:5d} "
            f"raw={diagnostics['fixed_raw_identity_mse']:.4f} "
            f"interval_cos={diagnostics['fixed_target_cosine_interval']:.4f} "
            f"boundary={diagnostics['boundary_velocity_mse']:.4f} "
            f"grad={grad_norm:.3f} "
            f"minutes={elapsed:.1f}"
        )
        model.train()

    if step in SAMPLE_STEPS:
        sample_snapshots[step] = generate_one_step(model).detach().cpu()
        write_samples_progress()
        model.train()

torch.save(
    {
        "model": model.state_dict(),
        "step": TRAIN_STEPS,
        "parameters": PARAMETERS,
    },
    CHECKPOINT_PATH,
)

print("training complete")


## 6. Result paths

학습이 끝나면 나한테 보내기 가장 좋은 것은 아래 **세 파일**이다.

- `samples_progress.png`
- `training_summary.png`
- `metrics.csv`

checkpoint는 재사용용이므로 결과 판독에는 필요 없다.


In [ ]:
# @title 6-1. Print result files and final metrics
print("samples_progress:", SAMPLES_PATH)
print("training_summary:", SUMMARY_PATH)
print("metrics_csv:", METRICS_PATH)
print("checkpoint:", CHECKPOINT_PATH)

print("\nFinal diagnostic row:")
for key, value in metric_rows[-1].items():
    print(f"{key}: {value}")
